In [ ]:
!pip install datasets -q
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

**LOAD AND EXPLORE THE DATASET**

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("jason23322/high-accuracy-email-classifier")
df_new = dataset['train'].to_pandas()

print("Shape:", df_new.shape)
print("\nColumns:", df_new.columns.tolist())
print("\nCategory Distribution:")
print(df_new['category'].value_counts())
print("\nSample:")
print(df_new.head(3))

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/4.15M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Shape: (10780, 6)

Columns: ['id', 'subject', 'body', 'text', 'category', 'category_id']

Category Distribution:
category
verify_code     1800
forum           1800
social_media    1796
promotions      1796
spam            1794
updates         1794
Name: count, dtype: int64

Sample:
               id                                    subject  \
0  promotions_582  Anniversary Special: Buy one get one free   
1       spam_1629         Your Amazon was used on new device   
2        spam_322                    Re: Your Google inquiry   

                                                body  \
0  As our loyal customer, get exclusive $60 off $...   
1  Your $5000 refund is processed. Claim: bit.ly/...   
2  Hi, following up about your Google application...   

                                                text    category  category_id  
0  Anniversary Special: Buy one get one free As o...  promotions            1  
1  Your Amazon was used on new device Your $5000 ...        spam           

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df_old = pd.read_pickle("/content/drive/MyDrive/email_data.pkl")
print("Enron Shape:", df_old.shape)
print("Enron Columns:", df_old.columns.tolist())

Mounted at /content/drive
Enron Shape: (15027, 5)
Enron Columns: ['file', 'message', 'clean_message', 'label', 'processed_text']


**COMBINE TWO DATASETS (HUGGING FACE + ENRON)**

In [ ]:
label_map = {
    'spam': 'Spam',
    'promotions': 'Promotions',
    'forum': 'Personal',
    'social_media': 'Personal',
    'updates': 'Support',
    'verify_code': 'Support'
}

df_new['label'] = df_new['category'].map(label_map)
df_new_clean = df_new[['text', 'label']].rename(columns={'text': 'processed_text'})

df_old_clean = df_old[['processed_text', 'label']]

df_combined = pd.concat([df_new_clean, df_old_clean], ignore_index=True)
df_combined = df_combined.dropna().reset_index(drop=True)

df_combined.to_pickle("/content/drive/MyDrive/email_combined.pkl")

print("Combined Shape:", df_combined.shape)
print("\nLabel Distribution:")
print(df_combined['label'].value_counts())
print("\nSample:")
print(df_combined.head(3))

Combined Shape: (25807, 2)

Label Distribution:
label
Personal      11366
Spam           6780
Support        4979
Promotions     2682
Name: count, dtype: int64

Sample:
                                      processed_text       label
0  Anniversary Special: Buy one get one free As o...  Promotions
1  Your Amazon was used on new device Your $5000 ...        Spam
2  Re: Your Google inquiry Hi, following up about...        Spam


**DATA PREPROCESSING**

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

In [ ]:
df_combined['processed_text'] = df_combined['processed_text'].apply(preprocess)

df_combined.to_pickle("/content/drive/MyDrive/email_combined.pkl")

print("Preprocessing done!")
print(df_combined[['processed_text', 'label']].head(3))

Preprocessing done!
                                      processed_text       label
0  anniversary special buy one get one free loyal...  Promotions
1  amazon used new device refund processed claim ...        Spam
2  google inquiry hi following google application...        Spam


**TRAIN/TEST SPLIT + CHECK DISTRIBUTION**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_combined['label_enc'] = le.fit_transform(df_combined['label'])

In [ ]:
print("Label Mapping:")
for i, c in enumerate(le.classes_):
    print(f"  {i} → {c}")

Label Mapping:
  0 → Personal
  1 → Promotions
  2 → Spam
  3 → Support


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_combined['processed_text'],
    df_combined['label_enc'],
    test_size=0.2,
    random_state=42,
    stratify=df_combined['label_enc']
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))
print("\nTrain Distribution:")
print(df_combined.loc[X_train.index, 'label'].value_counts())


Train size: 20645
Test size: 5162

Train Distribution:
label
Personal      9092
Spam          5424
Support       3983
Promotions    2146
Name: count, dtype: int64


**FINE-TUNE DISTILLBERT**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import classification_report
import torch
import numpy as np

In [ ]:
# Dataset class
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=128)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [ ]:
# Load DistilBERT
ft_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
ft_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Prepare datasets
train_dataset = EmailDataset(X_train.tolist(), y_train.tolist(), ft_tokenizer)
test_dataset = EmailDataset(X_test.tolist(), y_test.tolist(), ft_tokenizer)


In [ ]:
# Train
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.204913,0.164898
2,0.074255,0.110806
3,0.033374,0.116733


TrainOutput(global_step=1938, training_loss=0.16642726331918478, metrics={'train_runtime': 685.3119, 'train_samples_per_second': 90.375, 'train_steps_per_second': 2.828, 'total_flos': 2051165240570880.0, 'train_loss': 0.16642726331918478, 'epoch': 3.0})

In [ ]:
# Evaluate
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_test.tolist(), y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

    Personal       0.98      0.99      0.99      2274
  Promotions       0.98      0.93      0.95       536
        Spam       0.97      0.97      0.97      1356
     Support       0.98      0.97      0.97       996

    accuracy                           0.98      5162
   macro avg       0.98      0.97      0.97      5162
weighted avg       0.98      0.98      0.98      5162



**SAVE THE MODEL**

In [ ]:
import os, pickle

save_path = "/content/drive/MyDrive/email_classifier_v2/"
os.makedirs(save_path, exist_ok=True)

ft_model.save_pretrained(save_path + "distilbert_finetuned")
ft_tokenizer.save_pretrained(save_path + "distilbert_finetuned")
pickle.dump(le, open(save_path + "label_encoder.pkl", "wb"))

print("New model saved successfully!")
print(os.listdir(save_path))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New model saved successfully!
['distilbert_finetuned', 'label_encoder.pkl']
